## LANGCHAIN - PASO A PASO

### 1. CONECTAR EL LLM

In [ ]:
!pip install -qU langchain "langchain[google-genai]"


## Agente = Modelo + Arnés (Harness).
LangChain proporciona create_agent: un arnés mínimo y altamente configurable. El arnés es todo lo que rodea al bucle del modelo: el prompt, las herramientas y cualquier middleware que moldee el comportamiento. Comienza con las primitivas y compone exactamente lo que tu caso de uso necesita. Soporta OpenAI, Anthropic, Google y más."

------------------------------

## El Paradigma de "Agent Harness" en LangChain
### 1. Definición Clara y Técnica
En el desarrollo de aplicaciones de Inteligencia Artificial modernas, la arquitectura ha evolucionado desde los simples "wrappers" (envoltorios de código) hacia el concepto de Agent Harness (Arnés del Agente).

Técnicamente, un Harness es la infraestructura, entorno de ejecución y sistema de control que envuelve al ciclo de inferencia del LLM (el bucle del modelo). Mientras que el LLM funciona como el "motor cognitivo" aislado, el arnés actúa como el "chasis y los sistemas operativos" que le permiten interactuar de forma segura, estructurada y autónoma con el mundo real.

### 2. Capas de Ingeniería en el Ecosistema Actual
El desarrollo con LLMs se divide ahora en tres niveles concéntricos de ingeniería:

   1. **Prompt Engineering:** Diseñar las instrucciones de texto exactas que recibe el modelo.
   2. **Context Engineering:** Administrar qué datos específicos ve el modelo en su ventana de contexto y cuándo los ve.
   3. **Harness Engineering (El enfoque actual):** Engloba las dos anteriores y añade la orquestación de herramientas (Tool Calling), la persistencia del estado (memoria), la gestión de errores, los bucles de verificación y la seguridad del sistema.

### 3. ¿Por qué es la "Nueva Forma" de Conectar LLMs?
Anteriormente, conectar un LLM implicaba encadenar componentes de forma rígida (las antiguas Chains secuenciales de LangChain). El enfoque moderno basado en create_agent y abstracciones de arnés ofrece:

**Neutralidad de Modelos (Model Agnostic):** El arnés abstrae la API subyacente. Puedes intercambiar el motor (OpenAI, Anthropic, Google) sin reescribir la lógica de tus herramientas ni de tu memoria.

**Middleware Configurable:** Permite interceptar las peticiones y respuestas para inyectar lógica personalizada (ej. control de costos, compresión de historial o seguridad) antes y después de que el LLM actúe.

**Arquitectura de Bucle Abierto:** En lugar de ejecutar una sola petición, el arnés gestiona de manera autónoma un ciclo repetitivo: evalúa el prompt → llama al LLM → ejecuta herramientas si es necesario → procesa resultados → vuelve a preguntar al LLM.









In [ ]:
import os

from langchain.agents import create_agent
from google.colab import userdata

# Traer la API_KEY desde las variables de entorno (en esta caso desde un notebook en google colab)
# Esto evita exponer claves privadas al compartir el notebook (recuerde colocar la suya, en este caso la free de google)
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# 1. Inicializamos el arnés conversacional puro
agent = create_agent(
    model = "google_genai:gemini-3.6-flash",
    system_prompt="You are an expert Artificial Intelligence tutor. Provide concise and clear explanations. Answer in spanish",
)

# 2. El arnés se ejecuta enviando el estado de la conversación (messages)
# El usuario interactúa de forma directa mediante texto estructurado
result = agent.invoke(
    {
        "messages":
        [
            {
                "role": "user",
                "content": "Explain what an 'Agent Harness' is in one sentence."

            }
        ]
    }
)

# 3. Inspecionamos el Output de el LLM
# Recuperamos el último mensaje del historial retornado por el arnés.
last_message = result["messages"][-1]

# Opción A: Inspección de la estructura interna unificada (content_blocks)
print("--- Content Blocks Structure ---")
# Respuesta completa - print(last_message.content_blocks)
# Respuesta limpia
print(f"AI 🤖: {last_message.content_blocks[0].get("text", "")}")

# Opción B: Extracción del texto limpio (La recomendada para producción)
print("\n--- Final Text Output ---")
# Respuesta completa - print(last_message.content)
# Respuesta limpia
print(f"AI 🤖: {last_message.content[0].get("text", "")}")





--- Content Blocks Structure ---
AI 🤖: Un **Agent Harness** es la infraestructura de software que envuelve a un agente de IA para gestionar su ejecución, proporcionarle acceso a memoria y herramientas, y controlar de forma segura sus interacciones con el entorno.

--- Final Text Output ---
AI 🤖: Un **Agent Harness** es la infraestructura de software que envuelve a un agente de IA para gestionar su ejecución, proporcionarle acceso a memoria y herramientas, y controlar de forma segura sus interacciones con el entorno.


En la arquitectura moderna de LangChain, el método create_agent está diseñado con una filosofía minimalista: el arnés no debe duplicar los parámetros del modelo.

Por lo tanto, create_agent se divide en dos tipos de propiedades: los parámetros estructurales del arnés y los parámetros de generación de hiperparámetros (que se inyectan a través del string de configuración o diccionarios).

------------------------------
## 1. Parámetros de Estructura (Directos en create_agent)
Son las propiedades nativas que definen qué puede hacer el arnés del agente:


**model (str | BaseChatModel):** El identificador único ("google:gemini-2.5-flash", "openai:gpt-4o-mini").

**system_prompt (str | SystemMessage):** Define las reglas del sistema, el rol, las restricciones y el idioma de respuesta.

**tools (list[Callable | BaseTool]):** Lista de funciones nativas de Python que el agente puede invocar en su bucle operativo.

**response_format (BaseModel | TypedDict | dict):** (¡Muy importante en 2026!) Fuerza al agente a retornar un formato estructurado (usando una clase Pydantic) en lugar de texto plano, ideal para extraer JSON estructurado automáticamente.

**middleware (list):** Lista de capas o interceptores para auditar costos, inyectar seguridad, formatear el historial o detener ejecuciones inapropiadas.

------------------------------
## 2. Parámetros del Modelo (temperature, max_tokens, etc.)
Para modificar los parámetros cognitivos del LLM (hiperparámetros), no se pasan directamente a create_agent, sino que se configuran de dos maneras según el nivel del curso que estés dictando:

### Método A: Al inicializar el modelo (Para scripts locales avanzados)
Si en lugar de pasar un string pasas la instancia del objeto, puedes configurar el comportamiento exacto. Daremos un ejemplo de este metodo, a traves de las clases de integracion de langchain en este caso con google-genai
https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai

### Método B: Mediante Perfiles de Arnés (Harness Profiles)
En las versiones actuales de LangChain, puedes pasar diccionarios de configuración en el método .invoke() o registrar perfiles globales (HarnessProfile) para que el mismo string "google:gemini..." aplique restricciones dinámicas en caliente sin reconstruir el objeto. Este metodo tiene un nivel de complejidad medio-avanzado asi que se dejara para capitulos posteriores, si deseas leer al respecto:
https://learn.microsoft.com/en-us/agent-framework/concepts/agents/running-agents?pivots=programming-language-python

------------------------------

### Tabla Comparativa
Puedes añadir esta tabla resumida para tener una referencia rápida de qué controla cada parámetro:

| Parámetro | ¿Dónde se configura? | Tipo | ¿Qué controla en el Agente? |
|---|---|---|---|
| system_prompt | create_agent | String | La personalidad, rol e idioma base. |
| response_format | create_agent | Pydantic Class | Fuerza respuestas estructuradas (JSON con esquema). |
| temperature | Instancia del Modelo | Float (0 a 1) | Azar: 0.1 para códigos/datos, 0.8 para chats fluidos. |
| max_tokens | Instancia del Modelo | Integer | Longitud máxima permitida para la respuesta del modelo. |
| max_retries | Instancia del Modelo | Integer | Cuántas veces reintentar la llamada si hay error de red. |

### PARAMETROS COGNITIVOS (HIPERPARAMETROS)

**Método A: Al inicializar el modelo (Para scripts locales avanzados)**

In [ ]:
!pip install -U langchain-google-genai

In [36]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Aquí se configuran los hiperparámetros individuales del motor cognitivo
custom_model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.1,       # Más determinista (0.0) o más creativo (1.0)
    max_tokens=150,        # Límite estricto de tamaño de la respuesta
    top_p=0.95,            # Muestreo por núcleo (Nucleus sampling)
    max_retries=3          # Reintentos automáticos si la API de Google falla
)

agent = create_agent(
    model = custom_model,
    system_prompt="You are an expert Artificial Intelligence tutor. Provide concise and clear explanations. Answer in spanish",
)

result = agent.invoke(
    {
        "messages":
        [
            {
                "role": "user",
                "content": "Explain what an 'Agent Harness' is in one sentence."

            }
        ]
    }
)

print("\n--- Final Text Output ---")
# Respuesta completa - print(last_message.content)
# Respuesta limpia
print(f"AI 🤖: {last_message.content[0].get("text", "")}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(



--- Final Text Output ---
AI 🤖: Un **Agent Harness** es la infraestructura de software que envuelve a un agente de IA para gestionar su ejecución, proporcionarle acceso a memoria y herramientas, y controlar de forma segura sus interacciones con el entorno.
